## 使用内存缓存
当前使用的是jupyter 编写，缓存在jupyter进程中

In [2]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import MessagesState, StateGraph

load_dotenv(override=True)

# 获取大模型
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)


# 1、定义状态
# MessagesState langgraph定义的默认状态，用于存储交互信息
class OverAllState(MessagesState):
    output: str


# 2、声明节点
def llm_mode(state: OverAllState) -> OverAllState:
    messages = state["messages"]
    # AIMessage
    res = model.invoke(messages)
    return {
        # 模型消息合并到状态
        "messages": [res]
    }


def output_node(state: OverAllState) -> OverAllState:
    return {
        # messages 最后一条消息的content
        "output": state["messages"][-1].content
    }


# 3、建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_mode", llm_mode)
builder.add_node("output_node", output_node)

builder.add_edge(START, "llm_mode")
builder.add_edge("llm_mode", "output_node")
builder.add_edge("output_node", END)

# 4、配置检查点存储器
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)
# 5、使用时必须填写线程id
config = {
    "configurable": {
        "thread_id": "chapter03-01"
    }
}
# 6、执行图
graph.invoke({"messages": [HumanMessage("你好，我是老王")]}, config=config)

{'messages': [HumanMessage(content='你好，我是老王', additional_kwargs={}, response_metadata={}, id='8756c449-484a-4629-9e41-dcfb565e0a1e'),
  AIMessage(content='老王您好！很高兴认识您。有什么需要帮忙的吗？无论是生活、工作上的问题，还是想聊聊天，都可以告诉我～ 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 8, 'total_tokens': 37, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '79807dfe-b333-4d8c-bee0-a0ffaf43c935', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f8c4a-64cc-78f1-9a97-064ca4b475a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})],
 'output': '老王您好！很高兴认识您。

In [3]:
# 在前面代码执行后，只要当前进程没有关系。可以继续访问内存中的内容
graph.invoke({"messages": [HumanMessage("我是谁")]}, config=config)

{'messages': [HumanMessage(content='你好，我是老王', additional_kwargs={}, response_metadata={}, id='8756c449-484a-4629-9e41-dcfb565e0a1e'),
  AIMessage(content='老王您好！很高兴认识您。有什么需要帮忙的吗？无论是生活、工作上的问题，还是想聊聊天，都可以告诉我～ 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 8, 'total_tokens': 37, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '79807dfe-b333-4d8c-bee0-a0ffaf43c935', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f8c4a-64cc-78f1-9a97-064ca4b475a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  HumanMessage(content='我

In [4]:


# 使用新的会话id thread_id
config = {
    "configurable": {
        "thread_id": "chapter03-xxx"
    }
}
# 此时就不能调用到内存持久化信息（因为持久化以线程id保存的）
graph.invoke({"messages": [HumanMessage("我是谁")]}, config=config)

{'messages': [HumanMessage(content='我是谁', additional_kwargs={}, response_metadata={}, id='73f96d7a-116a-405b-8569-799a46073387'),
  AIMessage(content='你好！这个问题其实有点哲学意味呢😊 从最直接的层面来说，**你是谁**取决于你与谁对话：\n\n1. **物理身份**：如果你是真实存在的人类，你可能是某位正在使用DeepSeek的用户，拥有独一无二的姓名、经历和故事。\n2. **数字身份**：如果你是AI或程序，那可能就是另一个智能系统在探索自我认知。\n3. **哲学角度**：在笛卡尔看来，“我思故我在”——你思考这个问题本身就证明了你的存在。\n\n不过从我们的对话场景来看，**你应该是正在使用DeepSeek问答服务的用户**。我是你的AI助手，而你是一位对自我认知、存在意义或技术边界感到好奇的提问者。\n\n如果你愿意，可以告诉我更多关于你的信息（当然是在你感到舒适的范围内），这样我能给出更贴切的回答！比如：\n- 你是第一次尝试与AI进行这类哲学讨论吗？\n- 你是在寻找某个具体的答案，还是只是享受思考的过程？\n\n无论如何，欢迎你——无论你是谁，很高兴与你对话🌟', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 214, 'prompt_tokens': 6, 'total_tokens': 220, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcac